In [ ]:
!pip install -q executorch transformers sentencepiece pandas scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/BHARAT_AI_DATASET'
SRC  = f'{BASE}/kavach_indicbertV2'

import torch, os, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained(SRC, keep_accents=True, use_fast=False)
hf  = AutoModelForSequenceClassification.from_pretrained(SRC, attn_implementation="eager").eval()

class Kavach(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

model = Kavach(hf).eval()
SEQ = 128
def enc(t):
    e = tok(t, truncation=True, padding="max_length", max_length=SEQ, return_tensors="pt")
    return (e["input_ids"], e["attention_mask"])
example = enc("Your KYC is pending, account will be blocked")
print("Params:", sum(p.numel() for p in hf.parameters())/1e6, "M")

Mounted at /content/drive


W1109 10:51:18.462000 3236 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1109 10:51:18.520000 3236 torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Params: 33.445122 M


In [ ]:
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

ep = torch.export.export(model, example)
et = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch()
with open('/content/kavach_fp32.pte', 'wb') as f: f.write(et.buffer)
print("FP32 .pte:", os.path.getsize('/content/kavach_fp32.pte')/1e6, "MB")

FP32 .pte: 133.914376 MB


In [ ]:
try:
    from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
except ImportError:
    from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
try:
    from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
except ImportError:
    from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e

q = XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True, is_dynamic=True))
m = torch.export.export(model, example).module()
m = prepare_pt2e(m, q)
for s in ["KYC update now or account blocked", "நாளைக்கு காலேஜ் லீவா?", "Rs 500 debited from A/c XX1234"]:
    m(*enc(s))
m = convert_pt2e(m)

ep_q = torch.export.export(m, example)
et_q = to_edge_transform_and_lower(ep_q, partitioner=[XnnpackPartitioner()]).to_executorch()
with open('/content/kavach_int8.pte', 'wb') as f: f.write(et_q.buffer)
print("INT8 .pte:", os.path.getsize('/content/kavach_int8.pte')/1e6, "MB")

INT8 .pte: 110.678024 MB


In [ ]:
import pandas as pd
from executorch.runtime import Runtime
rt = Runtime.get()

def run_pte(path):
    method = rt.load_program(path).load_method("forward")
    res = {}
    for name in ["challenge_set.csv", "challenge_set_2.csv"]:
        ch = pd.read_csv(f'{BASE}/{name}', encoding='utf-8-sig')
        preds = [int(method.execute(list(enc(t)))[0].argmax()) for t in ch.text]
        res[name] = (pd.Series(preds) == ch.label).mean()
    x = list(enc("Your SBI account will be blocked today"))
    for _ in range(5): method.execute(x)
    t0 = time.perf_counter()
    for _ in range(50): method.execute(x)
    ms = (time.perf_counter() - t0) / 50 * 1000
    return res, ms

for p in ['/content/kavach_fp32.pte', '/content/kavach_int8.pte']:
    res, ms = run_pte(p)
    print(f"{os.path.basename(p):<18} {os.path.getsize(p)/1e6:6.1f} MB | set1 {res['challenge_set.csv']:.2f} | set2 {res['challenge_set_2.csv']:.2f} | {ms:.1f} ms/msg")

kavach_fp32.pte     133.9 MB | set1 0.90 | set2 0.90 | 308.0 ms/msg
kavach_int8.pte     110.7 MB | set1 0.89 | set2 0.91 | 254.5 ms/msg


In [ ]:
!mkdir -p {BASE}/pte
!cp /content/kavach_fp32.pte /content/kavach_int8.pte {BASE}/pte/
!ls -lh {BASE}/pte

total 234M
-rw------- 1 root root 128M Nov  9 10:58 kavach_fp32.pte
-rw------- 1 root root 106M Nov  9 10:58 kavach_int8.pte


In [ ]:
import glob, numpy as np
texts = pd.concat([pd.read_csv(p, encoding='utf-8-sig', on_bad_lines='skip') for p in glob.glob(f'{BASE}/DATA/*.csv')])['text'].astype(str).tolist()
lens = np.array([len(tok(t)['input_ids']) for t in texts])
for p in [50, 90, 95, 99, 100]:
    print(f"{p}th percentile: {int(np.percentile(lens, p))} tokens")

50th percentile: 25 tokens
90th percentile: 35 tokens
95th percentile: 37 tokens
99th percentile: 42 tokens
100th percentile: 53 tokens


In [ ]:
SEQ = 64   # change if Cell 7's 99th percentile is above 64

used = set()
for t in texts: used.update(tok(t)['input_ids'])
keep = sorted(set(range(50000)) | used | set(tok.all_special_ids))
print("Vocab:", tok.vocab_size, "->", len(keep))

hf2 = AutoModelForSequenceClassification.from_pretrained(SRC, attn_implementation="eager").eval()
old_emb = hf2.albert.embeddings.word_embeddings
new_emb = torch.nn.Embedding(len(keep), old_emb.embedding_dim)
new_emb.weight.data = old_emb.weight.data[keep].clone()
hf2.albert.embeddings.word_embeddings = new_emb
hf2.config.vocab_size = len(keep)

old2new = torch.full((tok.vocab_size,), keep.index(tok.unk_token_id), dtype=torch.long)
old2new[torch.tensor(keep)] = torch.arange(len(keep))

class KavachSmall(torch.nn.Module):
    def __init__(self, m, mp):
        super().__init__(); self.m = m; self.register_buffer("mp", mp)
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=self.mp[input_ids], attention_mask=attention_mask).logits

small = KavachSmall(hf2, old2new).eval()
example = enc("Your KYC is pending, account will be blocked")
print("Params:", sum(p.numel() for p in hf2.parameters())/1e6, "M")

m = torch.export.export(small, example).module()
m = prepare_pt2e(m, q)
for s in texts[:200]: m(*enc(s))
m = convert_pt2e(m)
et_s = to_edge_transform_and_lower(torch.export.export(m, example), partitioner=[XnnpackPartitioner()]).to_executorch()
with open('/content/kavach_int8_small.pte', 'wb') as f: f.write(et_s.buffer)

res, ms = run_pte('/content/kavach_int8_small.pte')
print(f"kavach_int8_small  {os.path.getsize('/content/kavach_int8_small.pte')/1e6:6.1f} MB | set1 {res['challenge_set.csv']:.2f} | set2 {res['challenge_set_2.csv']:.2f} | {ms:.1f} ms/msg")
!cp /content/kavach_int8_small.pte {BASE}/pte/

Vocab: 200000 -> 53904


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Params: 14.744834 M
kavach_int8_small    37.5 MB | set1 0.88 | set2 0.90 | 111.2 ms/msg
